## LSM pricer

### Simulating GBM (as described in Chapter 4 in the report)

In [ ]:
import numpy as np
import torch
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
import time

np.random.seed(7)
torch.manual_seed(7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float32

#Simulating GBM (as described in Chapter 4 in the report)
def simulate_gbm(S0, d, N, K, r, q, sigma, rho, T, n_dates,
                 device = device, dtype = dtype):

    dt = T/n_dates
    
    S = torch.empty((N,n_dates + 1, d), device = device, dtype = dtype)
    S[:,0,:] = S0

    pi = torch.full((d,d), rho, device = device, dtype = dtype)
    pi.fill_diagonal_(1.0)
    L = torch.linalg.cholesky(pi)                   #cholesky is lower triang decomposition where pi = L*L^T

    for i in range(n_dates):
        z = torch.randn((N,d), device = device, dtype = dtype)
        omega = z @ L.T                              #var(z*L^T) = Lvar(z)LT = L*I*L^T = L*L^T = var(omega)
        S[:,i+1,:] =  S[:,i,:]*torch.exp((r-q-0.5*sigma**2)*dt + sigma*torch.sqrt(torch.tensor(dt, device = device, dtype = dtype))*omega)

    return S


def payoff_max_call(S_t, K):
    return torch.relu(S_t.max(dim = -1).values - K) 

### LSM backward recursion algorithm (Algorithm 1 in Chapter 1 at the end)

In [ ]:
def lsm_maxcall_pricer(paths, N, d, K, T, n_dates, r, degree = 2):

    dt = T/n_dates

    v_cont = payoff_max_call(paths[:,n_dates,:], K = K)                        #value of the option at time T for each pah

    basis = PolynomialFeatures(degree = degree,include_bias = True)
    lr = LinearRegression(fit_intercept = False)

    for i in range(n_dates - 1,0, -1):                                          #backwards recursion
        X_i = paths[:,i,:]
        y_i = v_cont

        X_np = X_i.detach().cpu().numpy()                                        #convert to numpy
        y_np = y_i.detach().cpu().numpy()

        phi = basis.fit_transform(X_np)
        lr.fit(phi, y_np)

        cont_est_np = lr.predict(phi)
        cont_est = torch.tensor(cont_est_np, device = device, dtype = dtype)     #estimated continuation value

        c_hat = np.exp(-r*dt) * cont_est                                        #estimated discounted continuation value

        g_i = payoff_max_call(X_i, K = K)                                       #payoff at time i
        exer = (g_i > c_hat) & (g_i > 0)                                        #exercise if payoff > estimated continuation value and payoff>0

        v_cont = torch.where(exer, g_i, np.exp(-r*dt) * v_cont)                 #update cashflows

    v0 = np.exp(-r*dt) * v_cont.mean().item()
    
    return v0

### Compute option value + computational time

In [ ]:
S0 = 110.0             #spot price
T = 3.0                #maturity
K = 100.0              #strike price
n_dates = 9            #num of exercise dates
r = 0.05               #risk free interest rate
q = 0.1                #divident yield
sigma = 0.2            #volatility
rho = 0.0              #correlation
N = 10000              #number of simulated paths
d = 50                 #number of underlyings

paths = simulate_gbm(S0 = S0, d = d, N = N, K = K, r = r, q = q, sigma = sigma, rho = rho,
                     T = T, n_dates = n_dates,
                     device = device, dtype = dtype)

start = time.time()

v = lsm_maxcall_pricer(paths = paths, N = N, d = d, K = K, T = T, n_dates = n_dates, r = r, degree = 2)

stop = time.time()


print( f"d = {d} | S0 = {S0} | {v:.3f} | time: {stop - start:.2f}s")